# 把科学观测写成 token，交给同一个 nanoGPT

选一个方向，依次运行下面五格：看一条样本被切成什么 token，用不同数量的无标签样本预训练，用 32 个标签训练下游读出，最后把预训练损失和下游误差画在一起。默认方向在普通 CPU 上几分钟跑完，不需要 GPU。

可选方向：spectrum（光谱）、protein（蛋白质）、dna、rna、smiles（小分子）、crystal（晶体）、structure3d（三维结构）、weather（天气场）、field（扩散场）、image（天文图像）、text（文本）。

In [ ]:
FIELD = "field"
SIZES = [200, 800, 3200]   # 预训练用的无标签对象数，可以加上 12800
SEED = 53

import importlib.util
import subprocess
import sys
import time
from pathlib import Path

if importlib.util.find_spec("torch") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "torch",
                           "--index-url", "https://download.pytorch.org/whl/cpu"])
missing = [pip for module, pip in (("numpy", "numpy"), ("sklearn", "scikit-learn"), ("matplotlib", "matplotlib"))
           if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

here = Path.cwd().resolve()
root = next((p for p in [here, *here.parents] if (p / "nanoscigpt" / "tasks" / "eleven.py").exists()), None)
if root is None:
    subprocess.check_call(["git", "clone", "--depth", "1", "https://github.com/Yu-Yang-Li/nanoSciGPT.git"])
    root = here / "nanoSciGPT"
sys.path.insert(0, str(root))

import torch
torch.set_num_threads(2)
from nanoscigpt.tasks import eleven

spec = eleven.FIELDS[FIELD]
print(f"{spec['name']}：一个 token = {spec['token']}")
print(f"下游任务：{spec['task']}；读出：{'冻结主干 + 线性头' if spec['readout'] == 'probe' else '全参数微调'}；"
      f"预训练：{'每档 1,600 步' if spec['budget'] == 'fixed' else '每个对象平均 32 遍'}")

In [ ]:
parts, task = eleven.load(FIELD)
inputs, labels = task["A"]
eleven.show_tokens(FIELD, {name: values[0] for name, values in inputs.items()})
print("这条样本的下游标签：", round(float(labels[0]), 4))

In [ ]:
parts, task = eleven.normalize(FIELD, parts, task, SEED)
models, pretrain_loss = {}, {}
for size in [0, *SIZES]:
    start = time.perf_counter()
    models[size] = eleven.pretrain(FIELD, parts, size, SEED)
    if size:
        pretrain_loss[size] = eleven.heldout_loss(models[size], parts["A"][0])
        print(f"{size:>6} 个无标签对象：留出样本上的下一 token 损失 {pretrain_loss[size]:.4f}，用时 {time.perf_counter() - start:.0f} 秒")

In [ ]:
downstream_error = {}
for size, model in models.items():
    downstream_error[size], guess_mean = eleven.downstream(FIELD, model, task, SEED)
    name = "不预训练" if size == 0 else f"{size} 个对象预训练"
    print(f"{name:>14}：下游平均绝对误差 {downstream_error[size]:.4f}")
print(f"{'只猜平均值':>14}：{guess_mean:.4f}")

In [ ]:
import matplotlib
import matplotlib.pyplot as plt

installed = {f.name for f in matplotlib.font_manager.fontManager.ttflist}
chinese = [f for f in ("Microsoft YaHei", "SimHei", "Noto Sans CJK SC", "WenQuanYi Zen Hei", "PingFang SC") if f in installed]
plt.rcParams.update({"font.family": chinese[:1] or ["DejaVu Sans"], "axes.unicode_minus": False})

sizes = list(pretrain_loss)
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(sizes, [pretrain_loss[s] / pretrain_loss[sizes[0]] for s in sizes], "^--", label="预训练损失（留出）")
ax.plot(sizes, [downstream_error[s] / downstream_error[sizes[0]] for s in sizes], "o-", label="下游误差（留出）")
ax.axhline(downstream_error[0] / downstream_error[sizes[0]], color="grey", ls=":", label="下游：不预训练")
ax.set_xscale("log")
ax.set_xticks(sizes, [str(s) for s in sizes])
ax.set_xlabel("预训练无标签对象数")
ax.set_ylabel(f"相对 {sizes[0]} 档（越低越好）")
ax.set_title(f"{spec['name']} · {spec['task']}")
ax.legend()
plt.show()